# MV results overview

This notebook summarises one MV variant of an LV version. The default view shows a run and electrical
health table, a few totals, a region map, one district and ring, and the main distributions. Set
`show_details = True` for the full regional, district, and diagnostic tables and additional figures.
Use `2_mv_variant_comparison.ipynb` to compare all variants.

The MV stage (`pylovo-mv generate`, see `0_mv_generation.ipynb`) stores per MV variant of an LV version:

- `pylovo.mv_grid`: one row per district (an HV/MV substation and its supply area) with counts, lengths and the
  electrical extremes of the load, (n-1) and feed-in cases;
- `pylovo.mv_node` and `pylovo.mv_line`: busbars, MV/LV stations, MV customers, MV generators and joints; ring,
  stub, dedicated, generator and tie lines;
- `pylovo.mv_ring`: one row per open ring, dedicated feeder, generator feeder or busbar radial;
- `pylovo.mv_substation` and `pylovo.mv_variant` (parameter snapshot and run summary).

**Prerequisites**

- pylovo with the `notebooks` extra (`uv sync --extra notebooks`) and this notebook running on the kernel of the
  project environment (`.venv`), see `notebook_tutorials/README.md`.
- A `.env` in the repository root that points to the database with the MV results, and MV variants of the LV
  version (`pylovo-mv list --version <version>`; `0_mv_generation.ipynb` shows how to generate them).
- The notebook only reads from the database: its client runs read-only transactions.

**Region parts and filter.** The planning region of a variant is the union of its PLZ polygons. Groups of PLZ
more than 5 km apart are separate region parts, numbered by area (1 = largest). A district belongs to the part
nearest to its substation. `regions` and `plz` keep whole districts.

In [ ]:
# Parameters
version_id = "mv2"  # existing review results; None: VERSION_ID of config/config_generation_lv.yaml
variants = None    # None: load1, else d1, else first available; e.g. ["d0", "d1"] selects explicitly
show_details = False  # full per-region, per-class, per-district and diagnostic output
regions = None     # region parts to keep (1 = largest), e.g. [1]; None: all
plz = None         # keep only the districts with nodes in these PLZ; None: all
grid_id = None     # district of detail maps in the first mapped variant; None: most stations
map_variants = None  # None: selected variants; []: no maps; otherwise a subset of variants
basemap = False    # OpenStreetMap tiles under the maps (needs internet access)
dbname = "infdb_mv"  # verify the isolated MV review database
repo_root = None   # repository root; None: two levels above this notebook


## Setup

The first cell changes to the repository root, so that pylovo reads the `.env` and `config/` of this checkout
(and writes its log to `log/`), and checks the database name when `dbname` is set. `mv_notebook_utils.py` next to
this notebook holds the shared helpers of the MV notebooks.

In [ ]:
import os
import sys
from pathlib import Path

if repo_root is None:
    here = Path.cwd().resolve()
    repo_root = next((p for p in (here, *here.parents)
                      if (p / "pyproject.toml").is_file() and (p / "src" / "pylovo").is_dir()), None)
    if repo_root is None:
        raise RuntimeError("Set repo_root to the MV checkout before running this notebook.")
repo_root = Path(repo_root).expanduser().resolve()

# A kernel registered for another checkout can silently import its pylovo installation.
import importlib.util

spec = importlib.util.find_spec("pylovo")
expected = repo_root / "src" / "pylovo" / "__init__.py"
if spec is None or spec.origin is None or Path(spec.origin).resolve() != expected:
    raise RuntimeError(
        f"Select the notebook kernel at {repo_root / '.venv' / 'bin' / 'python'}, "
        "restart it, then Run All. The current kernel imports pylovo from "
        f"{None if spec is None else spec.origin!r}."
    )
os.chdir(repo_root)
sys.path.insert(0, str(repo_root / "notebook_tutorials" / "mv_grids"))

from pylovo import config_loader

if dbname is not None:
    assert config_loader.DBNAME == dbname, f"the .env points to {config_loader.DBNAME!r}, not {dbname!r}"

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

import mv_notebook_utils as mvu
from pylovo.mv_grid import metrics
from pylovo.plotting import mv_grid as mvplot
from pylovo.plotting.utils import get_color_map

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 400)
pd.set_option("display.max_colwidth", 200)

dbc = mvu.read_only_client()
version_id, variants, listing = mvu.resolve(dbc, version_id, variants, default_variant=("load1", "d1"))
print(f"Database {config_loader.DBNAME}, LV version {version_id}, selected MV variants: {', '.join(variants) or 'none'}")

In [ ]:
assert variants, "no MV variants to analyse"
plz_union = sorted({int(p) for p in listing.loc[listing["mv_variant_id"].isin(variants), "plz_list"].explode()})
parts = mvu.region_parts(dbc, plz_union)
data = {v: mvu.load_variant(dbc, version_id, v, parts=parts, regions=regions, plz=plz) for v in variants}
colors = get_color_map(variants)
districts = pd.concat([mvu.district_table(v) for v in data.values()], ignore_index=True)
rings = pd.concat([mvu.ring_table(v) for v in data.values()], ignore_index=True)
print(f"{len(parts)} region parts; districts per variant: "
      + ", ".join(f"{k} {len(v.grids)}" for k, v in data.items()))
shown = variants if map_variants is None else list(map_variants)
unknown_maps = [name for name in shown if name not in data]
if unknown_maps:
    raise ValueError(f"map_variants must be among selected variants: {unknown_maps}")


## Run and electrical health

The variant mode says which cases were requested. `generated` counts districts that finished construction. `all_cases_converged` counts districts where every requested
MV-only power-flow case converged; `cases_unknown` counts districts with missing case results.
`compliant` additionally requires a `converged` planning status and no recorded electrical violations.
`complete` describes geographic coverage; `complete_compliant` satisfies both geography and electrical rules.
Band conflicts and tap limits count affected stations, so one district can contribute many. `stale` means its
source LV grids changed after the run.


In [ ]:
cases = pd.concat([mvu.case_table(v) for v in data.values()], ignore_index=True)
case_flags = cases.groupby(["variant", "mv_grid_id"])["converged"].agg(
    all_cases_converged=lambda s: bool(s.notna().all() and s.eq(True).fillna(False).all()),
    cases_unknown=lambda s: bool(s.isna().any()),
)
health = []
for name in variants:
    d = districts[districts["variant"] == name].copy()
    row = listing.loc[listing["mv_variant_id"] == name].iloc[0]
    flags = case_flags.loc[name].reindex(d["mv_grid_id"])
    generated = d["status"].eq("done").to_numpy()
    all_converged = flags["all_cases_converged"].fillna(False).to_numpy(dtype=bool)
    no_violations = d["violations"].fillna("").eq("").to_numpy()
    planning_ok = d["power_flow_status"].eq("converged").to_numpy()
    compliant = generated & all_converged & planning_ok & no_violations
    complete = d["complete"].fillna(False).to_numpy(dtype=bool)
    health.append({
        "variant": name, "planning_scope": "load + N-1 + generation" if data[name].include_generation else "load + N-1",
        "run_status": row["status"], "districts": len(d),
        "generated": int(generated.sum()), "failed": int(d["status"].eq("failed").sum()),
        "all_cases_converged": int(all_converged.sum()),
        "cases_unknown": int(flags["cases_unknown"].fillna(True).sum()),
        "compliant": int(compliant.sum()), "complete": int(complete.sum()),
        "complete_compliant": int((complete & compliant).sum()),
        "with_violations": int((~no_violations).sum()),
        "band_conflicts": int(d["n_band_conflicts"].fillna(0).sum()),
        "tap_limits": int(d["n_tap_limit"].fillna(0).sum()), "stale": row["stale"],
    })
display(mvu.pretty(pd.DataFrame(health).set_index("variant")))
for h in health:
    name = h["variant"]
    d = districts[districts["variant"] == name]
    circuit_km = d["circuit_km"].sum()
    route_km = d["route_km"].sum()
    cable_share = d["cable_km"].sum() / circuit_km if circuit_km else np.nan
    print(f"{name}: {h['generated']}/{h['districts']} districts generated; "
          f"{h['all_cases_converged']} had every MV case converge, {h['compliant']} meet the requested MV criteria, "
          f"{h['complete_compliant']} are both geographically complete and compliant.")
    print(f"  Circuit inventory {circuit_km:,.0f} km (parallel circuits counted); "
          f"route length {route_km:,.0f} km (each routed line once); cable share {cable_share:.1%}.")
    if d["coupled_status"].notna().any():
        lv_snapshot = dbc.lv_snapshot(version_id)
        dbc.conn.rollback()
        lv_floor = (lv_snapshot.get("power_flow_assessment") or {}).get("min_vm_pu", 0.9)
        station_nodes = data[name].nodes[data[name].nodes["kind"].eq("station")]
        checked_lv = pd.to_numeric(station_nodes["coupled_lv_min_vm_pu"], errors="coerce").dropna()
        below_lv = int(checked_lv.lt(lv_floor).sum())
        print(f"  LV coupling: {int(d['coupled_status'].eq('done').sum())}/{len(d)} districts complete; "
              f"{int(d['coupled_status'].eq('partial').sum())} partial, "
              f"{int(d['coupled_status'].eq('failed').sum())} failed. "
              f"{below_lv}/{len(checked_lv)} checked LV nets fall below {lv_floor:g} p.u. at their design point.")
        if below_lv or d["coupled_status"].ne("done").any():
            print("  MV acceptance does not establish LV acceptance; see 4_mv_lv_coupling.ipynb.")
    if not data[name].include_generation:
        print("  Generation and feed-in were not modeled; these are not passed feed-in checks.")
    if h["failed"] or h["generated"] < h["districts"]:
        print("  Check failed or unfinished districts before using this run.")
    if h["compliant"] < h["districts"]:
        print("  Inspect case limits below and the violation table (show_details=True) before planning use.")
    if h["complete_compliant"] < h["complete"]:
        print("  Some complete districts still violate electrical rules; completeness alone is not a quality pass.")
    if h["band_conflicts"] or h["tap_limits"]:
        print("  Review station taps here and in 4_mv_lv_coupling.ipynb.")
if show_details:
    columns = ["mv_variant_id", "mv_variant_comment", "status", "n_plz", "n_grids", "n_done", "n_failed",
               "circuit_km", "cable_share", "stale", "finished_at"]
    display(mvu.pretty(listing.loc[listing["mv_variant_id"].isin(variants), columns].set_index("mv_variant_id")))
    runs = pd.DataFrame([{
        "variant": name, "stage version": v.params.get("mv_stage_version"), "status": v.summary.get("status"),
        "elapsed_s": v.summary.get("elapsed_s"), "n_districts_done": v.summary.get("grids_done"),
        "n_districts_failed": v.summary.get("grids_failed"), "n_plz_missing": len(v.summary.get("plz_missing") or []),
        "n_warnings": len(v.summary.get("warnings") or []),
        "ties": (v.summary.get("ties") or {}).get("n_ties"),
    } for name, v in data.items()]).set_index("variant")
    display(mvu.pretty(runs))
    for name, v in data.items():
        for warning in v.summary.get("warnings") or []:
            print(f"{name}: {warning}")
    display(mvu.parameter_table(list(data.values())))


## Optional detail: Region parts and HV/MV substations

The substations come from OSM (`power=substation` with 110 kV) or eGon-data (`MV_SUBSTATIONS`). A substation
without a district has no nodes (`no_nodes`) or too long a connection to the street graph (`long_attach`). `eGon
match`: an eGon-data substation lies within `egon_match_m` of the site.

In [ ]:
if show_details:
    display(mvu.pretty(parts[["region", "area_km2", "n_plz"]].set_index("region")))
    rows = []
    for name, v in data.items():
        s = v.substations
        flags = pd.Series([f for fl in s["flags"] for f in fl], dtype=object).value_counts()
        rows.append({"variant": name, "n_substations": len(s), "n_with_district": int(s["used"].sum()),
                     "n_in_region": int(s["in_region"].sum()), "source": ", ".join(sorted(set(s["source"]))),
                     "n_egon_match": int(s["egon_bus_id"].notna().sum()),
                     "hv_mv_units": ", ".join(f"{int(n)} x {m:g} MVA: {c}" for (n, m), c in v.grids.groupby(
                         ["hv_mv_transformers", "hv_mv_transformer_mva"]).size().items()),
                     "flags": ", ".join(f"{k} {c}" for k, c in flags.items())})
    display(mvu.pretty(pd.DataFrame(rows).set_index("variant")))


## Optional detail: Variant totals

Sums over the selected districts; ratios are formed from the sums. MV customers have an own MV node
(`n_customers`, `customer_mw` = their own peak, `n_dedicated_customers` on a dedicated feeder) or hang on the
LV busbar of the nearest station (`n_busbar_customers`, below `customer_min_mv_kw`). Ring statistics count the
open rings (`mv_ring.kind = ring`); `ring_members` are the stations, MV customers and looped-in generators of a
ring (the cap `MV_MAX_STATIONS_PER_RING`), half-rings run from the busbar to the normally-open point.
`km_per_km2_cells` divides by the district cells, `km_per_km2` clips the lines to the region parts. The demand
anchor is the coincident MV load (pylovo's coincidence law on all stations and MV customers) per inhabitant.
Ties (normally-open lines between districts) are not part of `circuit_km`.

In [ ]:
if show_details:
    anchors = pd.concat([mvu.demand_anchors(dbc, v) for v in data.values()], ignore_index=True)
    clipped = mvu.clipped_table(list(data.values()))
    totals = mvu.summary_table(districts, rings, by=["variant"])
    totals = totals.join(anchors[anchors["scope"] == "all"].set_index("variant")[["kw_per_inhabitant", "population"]])
    totals = totals.join(clipped.xs(0, level="region")[["km_per_km2"]])
    mvu.show_blocks(totals)


## Optional detail: Per region part

The same table per region part (region 0 of `km_per_km2` is all parts together, shown in the totals above).

In [ ]:
if show_details:
    by_region = mvu.summary_table(districts, rings, by=["variant", "region"])
    region_anchor = anchors[anchors["scope"] == "region"].assign(region=lambda a: a["key"].astype(int))
    by_region = by_region.join(region_anchor.set_index(["variant", "region"])[["kw_per_inhabitant", "population"]])
    by_region = by_region.join(clipped.drop(index=0, level="region")[["km_per_km2"]])
    mvu.show_blocks(by_region)


## Optional detail: Per area class

Districts count by the majority class of their supply points, rings by their own majority class (a rural district
can hold urban rings). The demand anchor per class uses the PLZ whose stations are mostly of that class.

In [ ]:
if show_details:
    by_class = mvu.summary_table(districts, rings, by=["variant", "area_class"])
    class_anchor = anchors[anchors["scope"] == "area_class"].rename(columns={"key": "area_class"})
    by_class = by_class.join(class_anchor.set_index(["variant", "area_class"])[["kw_per_inhabitant", "population"]])
    mvu.show_blocks(by_class)


## Power-flow cases

The MV-only district nets after the repairs in every case: the load case, the two (n-1) cases (each busbar exit
of every ring out in turn, the ring fed from the other side) and for generation-enabled variants, the two feed-in cases (wind and PV scaling of
SimBench). Per variant and case: districts converged, lowest and highest MV voltage, highest line loading, summed
slack power at the HV/MV busbars (negative: reverse flow) and unsupplied nodes.

In [ ]:
by_case = cases.groupby(["variant", "case"], sort=False).agg(
    n_districts=("mv_grid_id", "size"), n_converged=("converged", lambda c: int(c.eq(True).fillna(False).sum())),
    n_unknown=("converged", lambda c: int(c.isna().sum())),
    min_vm_pu=("min_vm_pu", "min"), max_vm_pu=("max_vm_pu", "max"),
    max_loading_percent=("max_loading_percent", "max"), slack_p_mw=("slack_p_mw", "sum"),
    n_unsupplied_nodes=("unsupplied_nodes", "sum"))
display(mvu.pretty(by_case))

## Optional detail: Districts

One row per district. `complete`: the substation lies in the planning region and the district cell is not cut
by the region border. `power_flow_status`: `converged` means no violation after the repairs.

In [ ]:
if show_details:
    columns = ["region", "area_class", "status", "power_flow_status", "complete", "substation_in_region", "area_km2",
               "hv_mv_units", "hv_mv_unit_mva", "peak_load_mw", "n_stations", "n_customers", "n_busbar_customers",
               "n_generators", "n_rings", "n_dedicated_feeders", "n_generator_feeders", "n_stub_nodes", "n_ties",
               "route_km", "circuit_km", "overhead_km", "cable_share", "km_per_station", "km_per_km2", "ring_members_mean",
               "ring_members_max", "half_ring_km_mean", "half_ring_km_max", "max_loading_percent",
               "max_loading_n1_percent", "max_loading_feed_in_percent", "max_drop_percent", "max_drop_n1_percent",
               "max_rise_percent", "n_repairs", "n_band_conflicts", "n_tap_limit", "runtime_s"]
    for name in variants:
        display(Markdown(f"**{name}**"))
        table = districts[districts["variant"] == name].set_index("mv_grid_id")[columns]
        display(mvu.pretty(table.sort_values(["region", "n_stations"], ascending=[True, False])))


## Maps

**Overview.** One panel per region part: district cells in one colour per district (hatched: incomplete),
substations as triangles, normally-open points as open red circles, overhead lines dashed.

In [ ]:
for name in shown:
    v = data[name]
    present = sorted(set(v.grids["region"]))
    fig, axes = plt.subplots(1, len(present), figsize=(10 * len(present), 10), squeeze=False)
    for ax, r in zip(axes[0], present):
        layers = mvu.region_layers(v, r)
        mvplot.plot_mv_overview(layers, ax=ax, basemap=basemap,
                                title=f"{name}, region {r}: {len(layers['grids'])} districts, "
                                      f"{layers['grids']['circuit_km'].sum():,.0f} km")
    fig.tight_layout()

**Optional detail (`show_details=True`): region ring maps.** Per region part, every district coloured per ring (the colours repeat between districts), cable solid,
overhead lines dashed, stub lines thinner, ties grey and dotted, district cells dashed, substations without a
district hollow.

In [ ]:
if show_details:
    for name in shown:
        v = data[name]
        for r in sorted(set(v.grids["region"])):
            fig = mvu.plot_ring_map(v, regions=[r], basemap=basemap, figsize=(12, 12))


**One district.** By ring and by line kind (cable, overhead, mixed), with optional line-loading and MV-voltage panels for the load case (`show_details=True`). The district is `grid_id` of the first mapped variant, or the one with the most
stations; the other variants show the district with the nearest substation.

In [ ]:
detail_of = {}
if shown:
    map_first = data[shown[0]]
    detail = grid_id if grid_id is not None else mvu.default_district(map_first)
    for name in shown:
        v = data[name]
        gid, distance = (detail, 0.0) if v is map_first else mvu.match_district(map_first, detail, v)
        detail_of[name] = gid
        fig, axes = plt.subplots(1, 2, figsize=(20, 10))
        for ax, mode in zip(axes, ("ring", "kind")):
            mvplot.plot_mv_grid_map(v.layers, gid, color_by=mode, ax=ax, basemap=basemap,
                                    title=f"{name}: district {gid} by {mode}")
        fig.tight_layout()


In [ ]:
if show_details:
    for name in shown:
        v = data[name]
        fig, axes = plt.subplots(1, 2, figsize=(20, 9))
        for ax, mode in zip(axes, ("loading", "voltage")):
            mvplot.plot_mv_grid_map(v.layers, detail_of[name], color_by=mode, ax=ax, settlements=False,
                                    title=f"{name}: district {detail_of[name]}, {mode} (load case)")
        fig.tight_layout()


**One ring.** The longest open ring of the district: its two halves from the busbar exits A and B, the
normally-open point and the members numbered along the ring; then its voltage profile in the load case and in
(n-1) (each half fed from the far exit) with the limits of the variant.

In [ ]:
for name in shown:
    v = data[name]
    gid = detail_of[name]
    candidates = v.rings[(v.rings["mv_grid_id"] == gid) & (v.rings["kind"] == "ring")]
    if candidates.empty:
        print(f"{name}: district {gid} has no open ring")
        continue
    ring_id = int(candidates.sort_values("length_km", ascending=False)["ring_id"].iloc[0])
    fig, axes = plt.subplots(1, 2, figsize=(20, 8.5))
    mvplot.plot_mv_ring(v.layers, gid, ring_id, ax=axes[0])
    mvplot.plot_voltage_profile(v.nodes, v.lines, gid, ring_id, ax=axes[1],
                                busbar_vm_pu=v.snapshot("voltage.busbar_vm_pu", 1.02),
                                max_drop_percent=v.snapshot("voltage.max_drop_percent", 5.0),
                                max_drop_n1_percent=v.snapshot("voltage.max_drop_n1_percent", 10.0))
    fig.tight_layout()

## Distributions

Ring members by area class (also half-ring and ring lengths with `show_details=True`) with the literature ranges of `pylovo.mv_grid.metrics.LITERATURE_RANGES`
(bands: ranges, dashed: one-sided bounds, dotted: typical values; see `3_mv_plausibility.ipynb`).

In [ ]:
ring_rows = rings[rings["kind"] == "ring"].rename(columns={"n_members": "ring_members", "length_km": "ring_km"})
halves = pd.concat([ring_rows[["variant", "area_class", c]].rename(columns={c: "feeder_km"})
                    for c in ("half_a_km", "half_b_km")], ignore_index=True)
distributions = (("ring_members", ring_rows), ("feeder_km", halves), ("ring_km", ring_rows))
for metric, frame in distributions if show_details else distributions[:1]:
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    for ax, cls in zip(axes, ("rural", "urban")):
        sub = frame[frame["area_class"] == cls]
        if sub.empty:
            ax.set_visible(False)
            continue
        mvplot.plot_metric_distribution(sub, metric, by="variant", ranges="auto", area_class=cls, ax=ax,
                                        order=[v for v in variants if v in set(sub["variant"])],
                                        title=f"{mvplot.metric_label(metric)}, {cls}")
    fig.tight_layout()

**Optional detail (`show_details=True`).** District values by the district's area class: circuit km per supply point, station spacing (median distance of
a station to its nearest station) and circuit km per km² of the district cell.

In [ ]:
if show_details:
    for metric in ("km_per_supply_point", "station_spacing_m", "km_per_km2"):
        fig, axes = plt.subplots(1, 2, figsize=(18, 6))
        for ax, cls in zip(axes, ("rural", "urban")):
            sub = districts[districts["area_class"] == cls]
            if sub.empty:
                ax.set_visible(False)
                continue
            mvplot.plot_metric_distribution(sub, metric, by="variant", ranges="auto", area_class=cls, ax=ax,
                                            order=[v for v in variants if v in set(sub["variant"])],
                                            title=f"{mvplot.metric_label(metric)} per district, {cls}")
        fig.tight_layout()


**Line loading** per case (ties left out; normally-open lines only in (n-1)), with the planning limit of the
first variant in normal operation and the (n-1) limit. Feed-in is plotted only for a generation-enabled selection.

In [ ]:
first = data[variants[0]]
lines_all = pd.concat([pd.DataFrame(v.lines.drop(columns=v.lines.geometry.name)).assign(variant=name)
                       for name, v in data.items()], ignore_index=True)
loading_cases = ("load", "n1", "feed_in") if any(v.include_generation for v in data.values()) else ("load", "n1")
fig, axes = plt.subplots(1, len(loading_cases), figsize=(7.3 * len(loading_cases), 6), squeeze=False)
for ax, case in zip(axes[0], loading_cases):
    frame = lines_all if case != "feed_in" else lines_all[lines_all["variant"].isin(
        [name for name, v in data.items() if v.include_generation])]
    mvplot.plot_loading_histogram(frame, case, by="variant", ax=ax,
                                  utilization=first.snapshot("loading.line_utilization", 0.6),
                                  n1_utilization=first.snapshot("loading.n1_utilization", 1.0))
fig.tight_layout()


**MV node voltages** per case (stations, MV customers and generators; busbar at its setpoint): load case,
the worse of the two (n-1) cases, and, only for generation-enabled variants, the larger of the two feed-in cases.

In [ ]:
nodes_all = pd.concat([pd.DataFrame(v.nodes.drop(columns=v.nodes.geometry.name)).assign(variant=name)
                       for name, v in data.items()], ignore_index=True)
members = nodes_all[nodes_all["kind"].isin(["station", "customer", "generator"])]
busbar = first.snapshot("voltage.busbar_vm_pu", 1.02)
limits = {"vm_load_pu": ("load case", busbar - first.snapshot("voltage.max_drop_percent", 5.0) / 100),
          "vm_n1_pu": ("(n-1) cases", busbar - first.snapshot("voltage.max_drop_n1_percent", 10.0) / 100)}
full_names = [name for name, v in data.items() if v.include_generation]
if full_names:
    full = data[full_names[0]]
    full_busbar = full.snapshot("voltage.busbar_vm_pu", 1.02)
    limits["vm_feed_in_pu"] = ("feed-in cases", full_busbar + full.snapshot("voltage.max_rise_percent", 2.0) / 100)
fig, axes = plt.subplots(1, len(limits), figsize=(7.3 * len(limits), 6), squeeze=False)
for ax, (column, (label, limit)) in zip(axes[0], limits.items()):
    values = pd.to_numeric(members[column], errors="coerce")
    names = full_names if column == "vm_feed_in_pu" else variants
    selected_values = values[members["variant"].isin(names)].dropna()
    if selected_values.empty:
        ax.text(0.5, 0.5, "No stored values", ha="center", va="center", transform=ax.transAxes)
        continue
    edges = np.histogram_bin_edges(selected_values, bins=40)
    for name in names:
        sub = values[members["variant"] == name].dropna()
        ax.hist(sub, bins=edges, histtype="step", linewidth=1.8, color=colors[name], label=f"{name} ({len(sub)})")
    ax.axvline(limit, color="#d62728", linestyle="--", linewidth=1.4, label=f"limit {limit:.3f} p.u.")
    ax.axvline(busbar, color="#2c3e50", linestyle=":", linewidth=1.2, label=f"busbar {busbar:.3f} p.u.")
    ax.set_xlabel(f"MV voltage, {label} (p.u.)")
    ax.set_ylabel("Nodes")
    ax.set_title(f"MV node voltages, {label}", fontweight="bold")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
fig.tight_layout()


## MV/LV taps and band conflicts

Every station gets the MV/LV tap that puts its LV busbar at the LV reference of the LV version at the design
point and, only when generation is enabled, keeps feed-in below `lv_feed_in_max_pu`. A **band conflict** in load-only mode means the active load-voltage budget cannot be met; in the full model it means no tap satisfies both load and feed-in; `tap_limit` a station at the end of the tap range. `4_mv_lv_coupling.ipynb` checks these
budgets with the LV nets.

In [ ]:
fig = mvplot.plot_tap_distribution({name: v.nodes for name, v in data.items()}, figsize=(12, 5))
taps = (nodes_all[nodes_all["kind"] == "station"]
        .assign(tap=lambda n: pd.to_numeric(n["mv_lv_tap_pos"], errors="coerce"),
                conflict=lambda n: n["band_conflict"].astype("boolean").fillna(False))
        .groupby(["variant", "region"])
        .agg(n_stations=("mv_node_id", "size"), n_band_conflicts=("conflict", "sum"),
             n_tap_nonzero=("tap", lambda t: int((t.fillna(0) != 0).sum())),
             tap_min=("tap", "min"), tap_max=("tap", "max")))
taps["band_conflict_share"] = taps["n_band_conflicts"] / taps["n_stations"]
display(mvu.pretty(taps))

In [ ]:
if show_details:
    from pylovo.mv_grid.parameters import LvVersionParameters

    lv = LvVersionParameters.from_generation_parameters(version_id, dbc.lv_snapshot(version_id))
    dbc.conn.rollback()
    print(f"LV reference voltage of version {version_id}: {lv.lv_reference_voltage_pu}")
    for name in shown:
        v = data[name]
        if not v.include_generation:
            print(f"{name}: feed-in voltage budget not modeled; see load-only tap and design checks above.")
            continue
        fig = mvplot.plot_voltage_budget(
            v.nodes, busbar_vm_pu=v.snapshot("voltage.busbar_vm_pu", 1.02),
            busbar_band_pu=v.snapshot("voltage.busbar_band_pu", 0.015),
            tap_step_percent=v.snapshot("taps.step_percent", 2.5),
            lv_reference_vm_pu=lv.lv_reference_voltage_pu or 0.96,
            lv_design_tolerance_pu=v.snapshot("taps.lv_design_tolerance_pu", 0.002),
            lv_feed_in_max_pu=v.snapshot("taps.lv_feed_in_max_pu", 1.07),
            title=f"{name}: voltage budget of {int((v.nodes['kind'] == 'station').sum())} stations")


## Optional detail: Repairs, violations and node flags

The electrical repair loop (at most `repair_rounds` rounds) converts T-off generators to dedicated feeders, adds
parallel cables or splits rings; the table counts its actions. Districts with violations after the repairs and
the node flags of the variants follow.

In [ ]:
if show_details:
    repairs = pd.concat([mvu.repairs_table(v) for v in data.values()], ignore_index=True)
    if repairs.empty:
        print("No repairs")
    else:
        display(repairs.groupby(["variant", "case", "violation", "action"]).size().rename("n").to_frame())
    violations = districts[districts["power_flow_status"] != "converged"]
    display(mvu.pretty(violations.set_index(["variant", "mv_grid_id"])[
        ["power_flow_status", "violations", "max_loading_percent", "max_loading_n1_percent", "max_drop_percent",
         "max_drop_n1_percent", "max_rise_percent", "repairs"]]))
    display(pd.concat([mvu.flag_counts(v) for v in data.values()], axis=1).fillna(0).astype(int))


## Optional detail: Demand anchor

The coincident MV load per inhabitant (`municipal_register.pop` of the PLZ), for all selected PLZ, per region
part and per area class; `approximate` when PLZ lie only partly in the region or have no population. The sum of
the district peaks per inhabitant is larger, because the district peaks are not coincident.

In [ ]:
if show_details:
    peaks = districts.groupby("variant")["peak_load_mw"].sum()
    table = anchors.copy()
    table["district_peaks_kw_per_inhabitant"] = [
        peaks[v] * 1000 / p if s == "all" and p else np.nan for v, s, p in zip(table["variant"], table["scope"],
                                                                               table["population"])]
    display(mvu.pretty(table.set_index(["variant", "scope", "key"])))
